In [ ]:
!pip install -q yfinance scikit-learn xgboost matplotlib seaborn pandas numpy

import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
from xgboost import XGBRegressor

# Plot styling
plt.rcParams.update({
    'figure.facecolor': '#0f0f1a',
    'axes.facecolor':   '#0f0f1a',
    'axes.edgecolor':   '#444',
    'axes.labelcolor':  '#ccc',
    'xtick.color':      '#ccc',
    'ytick.color':      '#ccc',
    'text.color':       '#eee',
    'grid.color':       '#2a2a3a',
    'grid.linestyle':   '--',
    'grid.alpha':       0.5,
})

print("✅ All libraries loaded successfully!")

In [ ]:

TICKER     = 'AAPL'
START_DATE = '2018-01-01'
END_DATE   = '2024-12-31'

print(f"📥 Downloading {TICKER} stock data from Yahoo Finance...")
df = yf.download(TICKER, start=START_DATE, end=END_DATE, progress=False)
df.dropna(inplace=True)

print(f"✅ Downloaded {len(df)} trading days of data")
print(f"\n{'='*45}")
print(f"  Stock     : {TICKER}")
print(f"  Period    : {df.index[0].date()} → {df.index[-1].date()}")
print(f"  Columns   : {list(df.columns)}")
print(f"{'='*45}")
print(f"\n  Opening Price  : ${float(df['Open'].iloc[0]):.2f}  →  ${float(df['Open'].iloc[-1]):.2f}")
print(f"  Closing Price  : ${float(df['Close'].iloc[0]):.2f}  →  ${float(df['Close'].iloc[-1]):.2f}")
print(f"  All-time High  : ${float(df['High'].max()):.2f}")
print(f"  All-time Low   : ${float(df['Low'].min()):.2f}")
df.head()

In [ ]:
fig = plt.figure(figsize=(18, 12))
fig.suptitle(f'{TICKER} — Exploratory Data Analysis', fontsize=16, fontweight='bold', color='white', y=1.01)
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

close = df['Close'].squeeze()
vol   = df['Volume'].squeeze()

# 1. Closing price over time
ax1 = fig.add_subplot(gs[0, :2])
ax1.plot(df.index, close, color='#00d4ff', linewidth=1.2, label='Close Price')
ax1.fill_between(df.index, close, alpha=0.08, color='#00d4ff')
ax1.set_title('Closing Price Over Time', fontweight='bold')
ax1.set_ylabel('Price (USD)')
ax1.legend()
ax1.grid(True)

# 2. Volume
ax2 = fig.add_subplot(gs[0, 2])
ax2.bar(df.index, vol, color='#f7931a', alpha=0.6, width=1)
ax2.set_title('Trading Volume', fontweight='bold')
ax2.set_ylabel('Volume')
ax2.grid(True)

# 3. Moving Averages
ax3 = fig.add_subplot(gs[1, :2])
ax3.plot(df.index, close, color='#888', linewidth=0.8, label='Close', alpha=0.6)
ax3.plot(df.index, close.rolling(20).mean(),  color='#00d4ff', linewidth=1.5, label='MA-20')
ax3.plot(df.index, close.rolling(50).mean(),  color='#ff6b6b', linewidth=1.5, label='MA-50')
ax3.plot(df.index, close.rolling(200).mean(), color='#ffd700', linewidth=1.5, label='MA-200')
ax3.set_title('Moving Averages', fontweight='bold')
ax3.set_ylabel('Price (USD)')
ax3.legend(fontsize=9)
ax3.grid(True)

# 4. Daily Returns distribution
ax4 = fig.add_subplot(gs[1, 2])
returns = close.pct_change().dropna()
ax4.hist(returns, bins=60, color='#a855f7', edgecolor='none', alpha=0.8)
ax4.axvline(returns.mean(), color='#ffd700', linestyle='--', linewidth=2, label=f'Mean: {returns.mean():.4f}')
ax4.set_title('Daily Returns Distribution', fontweight='bold')
ax4.set_xlabel('Daily Return')
ax4.legend(fontsize=9)
ax4.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  FEATURE ENGINEERING
#  We create technical indicators as input features for our ML models.
#  These are the same indicators used by real stock traders.
# ══════════════════════════════════════════════════════════════════════

data = df.copy()

# ── Price features ──────────────────────────────────────
data['Close']  = data['Close'].squeeze()
data['Open']   = data['Open'].squeeze()
data['High']   = data['High'].squeeze()
data['Low']    = data['Low'].squeeze()
data['Volume'] = data['Volume'].squeeze()

# ── Moving Averages ─────────────────────────────────────
#    Average price over N days. Smooths noise & shows trend.
data['MA_5']   = data['Close'].rolling(5).mean()
data['MA_20']  = data['Close'].rolling(20).mean()
data['MA_50']  = data['Close'].rolling(50).mean()

# ── Exponential Moving Average ───────────────────────────
#    Like MA but gives more weight to recent prices.
data['EMA_12'] = data['Close'].ewm(span=12, adjust=False).mean()
data['EMA_26'] = data['Close'].ewm(span=26, adjust=False).mean()

# ── MACD (Moving Average Convergence Divergence) ─────────
#    Momentum indicator. Shows buy/sell signals.
data['MACD']        = data['EMA_12'] - data['EMA_26']
data['MACD_Signal'] = data['MACD'].ewm(span=9, adjust=False).mean()
data['MACD_Hist']   = data['MACD'] - data['MACD_Signal']

# ── RSI (Relative Strength Index) ────────────────────────
#    Measures overbought (>70) or oversold (<30) conditions.
delta     = data['Close'].diff()
gain      = delta.clip(lower=0).rolling(14).mean()
loss      = (-delta.clip(upper=0)).rolling(14).mean()
rs        = gain / (loss + 1e-10)
data['RSI'] = 100 - (100 / (1 + rs))

# ── Bollinger Bands ───────────────────────────────────────
#    Price channels based on volatility.
bb_mid           = data['Close'].rolling(20).mean()
bb_std           = data['Close'].rolling(20).std()
data['BB_Upper'] = bb_mid + 2 * bb_std
data['BB_Lower'] = bb_mid - 2 * bb_std
data['BB_Width'] = data['BB_Upper'] - data['BB_Lower']

# ── Price momentum & volatility ───────────────────────────
data['Return_1d']  = data['Close'].pct_change(1)
data['Return_5d']  = data['Close'].pct_change(5)
data['Volatility'] = data['Return_1d'].rolling(10).std()

# ── Target: Next day's closing price ────────────────────
data['Target'] = data['Close'].shift(-1)  # Predict tomorrow's close

data.dropna(inplace=True)
print(f"✅ Feature engineering complete!")
print(f"   Total features : {data.shape[1] - 1}")
print(f"   Total samples  : {data.shape[0]} trading days\n")
print("Features created:")
FEATURES = ['Open','High','Low','Close','Volume',
            'MA_5','MA_20','MA_50','EMA_12','EMA_26',
            'MACD','MACD_Signal','MACD_Hist','RSI',
            'BB_Upper','BB_Lower','BB_Width',
            'Return_1d','Return_5d','Volatility']
for i, f in enumerate(FEATURES, 1):
    print(f"   {i:2d}. {f}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  TRAIN / TEST SPLIT
#  IMPORTANT: For time series data we NEVER shuffle randomly.
#  We always use the past to predict the future.
#  Training = first 80% of dates | Testing = last 20% of dates
# ══════════════════════════════════════════════════════════════════════

X = data[FEATURES].values
y = data['Target'].values
dates = data.index

SPLIT = int(len(X) * 0.80)
X_train, X_test = X[:SPLIT], X[SPLIT:]
y_train, y_test = y[:SPLIT], y[SPLIT:]
dates_test      = dates[SPLIT:]

# Normalize features to [0, 1] range
scaler   = MinMaxScaler()
X_train  = scaler.fit_transform(X_train)
X_test   = scaler.transform(X_test)

print(f"Training period : {dates[0].date()} → {dates[SPLIT-1].date()}  ({len(X_train)} days)")
print(f"Testing  period : {dates[SPLIT].date()} → {dates[-1].date()}   ({len(X_test)} days)")
print(f"\nFeature matrix shape : {X_train.shape}")
print(f"Target vector shape  : {y_train.shape}")

# Visualize the split
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(dates[:SPLIT], y_train, color='#00d4ff', linewidth=1.2, label=f'Training data ({len(X_train)} days)')
ax.plot(dates[SPLIT:], y_test,  color='#ff6b6b', linewidth=1.2, label=f'Test data ({len(X_test)} days)')
ax.axvline(x=dates[SPLIT], color='#ffd700', linestyle='--', linewidth=2, label='Train/Test boundary')
ax.set_title('Train / Test Split (Time-based — No shuffling)', fontweight='bold', fontsize=13)
ax.set_ylabel('Stock Price (USD)')
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
lr_pred  = lr_model.predict(X_test)

# Show feature weights (coefficients)
coefs = pd.DataFrame({'Feature': FEATURES,
                      'Weight':  lr_model.coef_}).sort_values('Weight', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Model 1: Linear Regression', fontsize=14, fontweight='bold', color='#00d4ff')

axes[0].plot(dates_test, y_test,  color='#aaa',     linewidth=1, label='Actual Price', alpha=0.8)
axes[0].plot(dates_test, lr_pred, color='#00d4ff',  linewidth=1.5, label='LR Prediction')
axes[0].set_title('Actual vs Predicted')
axes[0].set_ylabel('Price (USD)')
axes[0].legend(); axes[0].grid(True)

colors = ['#1D9E75' if w > 0 else '#E24B4A' for w in coefs['Weight']]
axes[1].barh(coefs['Feature'], coefs['Weight'], color=colors, edgecolor='none')
axes[1].set_title('Feature Weights\n(+ve = pushes price up | -ve = pushes price down)')
axes[1].axvline(0, color='white', linewidth=0.8)
axes[1].invert_yaxis()
axes[1].grid(True, axis='x')

plt.tight_layout()
plt.show()

lr_rmse = np.sqrt(mean_squared_error(y_test, lr_pred))
lr_mae  = mean_absolute_error(y_test, lr_pred)
lr_r2   = r2_score(y_test, lr_pred)
print(f"\n📊 Linear Regression Results:")
print(f"   RMSE (Root Mean Sq Error) : ${lr_rmse:.2f}  ← average dollar error")
print(f"   MAE  (Mean Abs Error)     : ${lr_mae:.2f}  ← average absolute error")
print(f"   R²   (Accuracy Score)     : {lr_r2:.4f} ← 1.0 = perfect")

In [ ]:
# Find optimal K
k_values = range(1, 21)
k_rmses  = []
for k in k_values:
    knn = KNeighborsRegressor(n_neighbors=k)
    knn.fit(X_train, y_train)
    pred   = knn.predict(X_test)
    k_rmses.append(np.sqrt(mean_squared_error(y_test, pred)))

best_k   = k_values[np.argmin(k_rmses)]
knn_model = KNeighborsRegressor(n_neighbors=best_k)
knn_model.fit(X_train, y_train)
knn_pred  = knn_model.predict(X_test)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Model 2: K-Nearest Neighbors (KNN)', fontsize=14, fontweight='bold', color='#a855f7')

axes[0].plot(dates_test, y_test,   color='#aaa',    linewidth=1,   label='Actual Price', alpha=0.8)
axes[0].plot(dates_test, knn_pred, color='#a855f7', linewidth=1.5, label=f'KNN Prediction (K={best_k})')
axes[0].set_title('Actual vs Predicted')
axes[0].set_ylabel('Price (USD)')
axes[0].legend(); axes[0].grid(True)

axes[1].plot(list(k_values), k_rmses, color='#a855f7', marker='o', markersize=5, linewidth=2)
axes[1].axvline(best_k, color='#ffd700', linestyle='--', label=f'Best K = {best_k}')
axes[1].set_title('Finding Best K Value\n(lower RMSE = better)')
axes[1].set_xlabel('Number of Neighbors (K)')
axes[1].set_ylabel('RMSE ($)')
axes[1].legend(); axes[1].grid(True)

plt.tight_layout()
plt.show()

knn_rmse = np.sqrt(mean_squared_error(y_test, knn_pred))
knn_mae  = mean_absolute_error(y_test, knn_pred)
knn_r2   = r2_score(y_test, knn_pred)
print(f"\n📊 KNN Results (Best K = {best_k}):")
print(f"   RMSE : ${knn_rmse:.2f}")
print(f"   MAE  : ${knn_mae:.2f}")
print(f"   R²   : {knn_r2:.4f}")

In [ ]:
svr_model = SVR(kernel='rbf', C=100, epsilon=0.1, gamma='scale')
svr_model.fit(X_train, y_train)
svr_pred  = svr_model.predict(X_test)

# Illustrate the epsilon-tube concept
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Model 3: Support Vector Regression (SVR)', fontsize=14, fontweight='bold', color='#f97316')

axes[0].plot(dates_test, y_test,  color='#aaa',    linewidth=1,   label='Actual Price', alpha=0.8)
axes[0].plot(dates_test, svr_pred,color='#f97316', linewidth=1.5, label='SVR Prediction')
axes[0].fill_between(dates_test, svr_pred - svr_model.epsilon,
                                  svr_pred + svr_model.epsilon,
                     alpha=0.15, color='#f97316', label=f'ε-tube (±{svr_model.epsilon})')
axes[0].set_title('Actual vs Predicted + ε-tube')
axes[0].set_ylabel('Price (USD)')
axes[0].legend(fontsize=9); axes[0].grid(True)

# Kernel concept visualization
x_demo = np.linspace(-3, 3, 200)
gammas = [0.1, 0.5, 1.0, 2.0]
for g in gammas:
    rbf = np.exp(-g * x_demo**2)
    axes[1].plot(x_demo, rbf, linewidth=2, label=f'γ = {g}')
axes[1].set_title('RBF Kernel — Similarity Function\n(How SVR measures closeness between points)')
axes[1].set_xlabel('Distance between points')
axes[1].set_ylabel('Similarity score')
axes[1].legend(); axes[1].grid(True)

plt.tight_layout()
plt.show()

svr_rmse = np.sqrt(mean_squared_error(y_test, svr_pred))
svr_mae  = mean_absolute_error(y_test, svr_pred)
svr_r2   = r2_score(y_test, svr_pred)
print(f"\n📊 SVR Results:")
print(f"   RMSE : ${svr_rmse:.2f}")
print(f"   MAE  : ${svr_mae:.2f}")
print(f"   R²   : {svr_r2:.4f}")
print(f"   Support vectors used : {svr_model.n_support_[0]} out of {len(X_train)} training points")

In [ ]:


rf_model = RandomForestRegressor(n_estimators=200, max_depth=10,
                                  min_samples_split=5, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
rf_pred  = rf_model.predict(X_test)

# Feature importances
importances = pd.Series(rf_model.feature_importances_, index=FEATURES).sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Model 4: Random Forest Regressor', fontsize=14, fontweight='bold', color='#1D9E75')

axes[0].plot(dates_test, y_test,  color='#aaa',    linewidth=1,   label='Actual Price', alpha=0.8)
axes[0].plot(dates_test, rf_pred, color='#1D9E75', linewidth=1.5, label='RF Prediction')
axes[0].set_title('Actual vs Predicted')
axes[0].set_ylabel('Price (USD)')
axes[0].legend(); axes[0].grid(True)

colors = plt.cm.YlGn(np.linspace(0.3, 0.9, len(importances)))
axes[1].barh(importances.index, importances.values, color=colors, edgecolor='none')
axes[1].set_title('Feature Importance\n(Which features matter most to the forest?)')
axes[1].set_xlabel('Importance Score')
axes[1].grid(True, axis='x')

plt.tight_layout()
plt.show()

rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
rf_mae  = mean_absolute_error(y_test, rf_pred)
rf_r2   = r2_score(y_test, rf_pred)
print(f"\n📊 Random Forest Results:")
print(f"   RMSE : ${rf_rmse:.2f}")
print(f"   MAE  : ${rf_mae:.2f}")
print(f"   R²   : {rf_r2:.4f}")
print(f"   Trees in forest : {rf_model.n_estimators}")
print(f"\n   Top 3 most important features:")
top3 = pd.Series(rf_model.feature_importances_, index=FEATURES).nlargest(3)
for feat, imp in top3.items():
    print(f"   → {feat:<15} : {imp:.4f} ({imp*100:.1f}% importance)")

In [ ]:



xgb_model = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                          subsample=0.8, colsample_bytree=0.8,
                          reg_alpha=0.1, reg_lambda=1.0,
                          random_state=42, verbosity=0)
xgb_model.fit(X_train, y_train,
              eval_set=[(X_test, y_test)],
              verbose=False)
xgb_pred = xgb_model.predict(X_test)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Model 5: XGBoost (Extreme Gradient Boosting)', fontsize=14, fontweight='bold', color='#f59e0b')

axes[0].plot(dates_test, y_test,   color='#aaa',    linewidth=1,   label='Actual Price', alpha=0.8)
axes[0].plot(dates_test, xgb_pred, color='#f59e0b', linewidth=1.5, label='XGBoost Prediction')
axes[0].set_title('Actual vs Predicted')
axes[0].set_ylabel('Price (USD)')
axes[0].legend(); axes[0].grid(True)

# Boosting concept illustration
np.random.seed(42)
x_demo  = np.linspace(0, 10, 50)
y_true  = np.sin(x_demo) + np.random.normal(0, 0.1, 50)
pred1   = np.zeros(50)
residuals = [y_true]
preds_cumulative = [pred1.copy()]
colors_boost = ['#f59e0b', '#ef4444', '#3b82f6', '#22c55e']
for i in range(4):
    resid = residuals[-1]
    correction = resid * 0.5 + np.random.normal(0, 0.05, 50)
    pred1 = pred1 + 0.3 * correction
    residuals.append(y_true - pred1)
    preds_cumulative.append(pred1.copy())
    axes[1].plot(x_demo, pred1, color=colors_boost[i], linewidth=1.5,
                 label=f'After tree {i+1}', alpha=0.8)
axes[1].plot(x_demo, y_true, 'w--', linewidth=2, label='True signal', alpha=0.5)
axes[1].set_title('Boosting: Each tree corrects previous errors\n(predictions get closer to truth)')
axes[1].legend(fontsize=8); axes[1].grid(True)

plt.tight_layout()
plt.show()

xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_pred))
xgb_mae  = mean_absolute_error(y_test, xgb_pred)
xgb_r2   = r2_score(y_test, xgb_pred)
print(f"\n📊 XGBoost Results:")
print(f"   RMSE : ${xgb_rmse:.2f}")
print(f"   MAE  : ${xgb_mae:.2f}")
print(f"   R²   : {xgb_r2:.4f}")

In [ ]:


try:
    import tensorflow as tf
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
    from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
    TF_AVAILABLE = True
    print(f"✅ TensorFlow {tf.__version__} available — running full LSTM")
except ImportError:
    TF_AVAILABLE = False
    print("⚠️  TensorFlow not found — using MLP as LSTM substitute")
    print("   Run:  !pip install tensorflow  then restart runtime")

LOOKBACK = 60  # Use 60 days of history to predict day 61

if TF_AVAILABLE:
    # Reshape data into 3D: [samples, timesteps, features]
    close_vals = data['Close'].values.reshape(-1, 1)
    price_scaler = MinMaxScaler()
    close_scaled = price_scaler.fit_transform(close_vals)

    def make_sequences(arr, lookback):
        X_s, y_s = [], []
        for i in range(lookback, len(arr)):
            X_s.append(arr[i-lookback:i, 0])
            y_s.append(arr[i, 0])
        return np.array(X_s), np.array(y_s)

    Xs, ys  = make_sequences(close_scaled, LOOKBACK)
    Xs      = Xs.reshape(Xs.shape[0], Xs.shape[1], 1)
    split_l = int(len(Xs) * 0.8)
    Xl_train, Xl_test = Xs[:split_l], Xs[split_l:]
    yl_train, yl_test = ys[:split_l], ys[split_l:]

    lstm_model = Sequential([
        LSTM(64, return_sequences=True, input_shape=(LOOKBACK, 1)),
        Dropout(0.2),
        LSTM(32, return_sequences=False),
        Dropout(0.2),
        Dense(16, activation='relu'),
        Dense(1)
    ])
    lstm_model.compile(optimizer='adam', loss='mse')
    lstm_model.summary()

    callbacks = [
        EarlyStopping(patience=10, restore_best_weights=True, monitor='val_loss'),
        ReduceLROnPlateau(factor=0.5, patience=5, monitor='val_loss')
    ]
    history = lstm_model.fit(Xl_train, yl_train, epochs=50, batch_size=32,
                             validation_split=0.1, callbacks=callbacks, verbose=1)

    lstm_pred_scaled = lstm_model.predict(Xl_test)
    lstm_pred = price_scaler.inverse_transform(lstm_pred_scaled).flatten()
    lstm_actual = price_scaler.inverse_transform(yl_test.reshape(-1,1)).flatten()

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    fig.suptitle('Model 6: LSTM Neural Network', fontsize=14, fontweight='bold', color='#ec4899')

    axes[0].plot(lstm_actual, color='#aaa',    linewidth=1,   label='Actual Price', alpha=0.8)
    axes[0].plot(lstm_pred,   color='#ec4899', linewidth=1.5, label='LSTM Prediction')
    axes[0].set_title('Actual vs Predicted')
    axes[0].set_ylabel('Price (USD)')
    axes[0].legend(); axes[0].grid(True)

    axes[1].plot(history.history['loss'],     color='#ec4899', label='Training Loss')
    axes[1].plot(history.history['val_loss'], color='#ffd700', label='Validation Loss')
    axes[1].set_title('LSTM Learning Curve\n(Loss should decrease = model is learning)')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('MSE Loss')
    axes[1].legend(); axes[1].grid(True)
    plt.tight_layout(); plt.show()

    lstm_rmse = np.sqrt(mean_squared_error(lstm_actual, lstm_pred))
    lstm_mae  = mean_absolute_error(lstm_actual, lstm_pred)
    lstm_r2   = r2_score(lstm_actual, lstm_pred)
    print(f"\n📊 LSTM Results:")
    print(f"   RMSE : ${lstm_rmse:.2f}")
    print(f"   MAE  : ${lstm_mae:.2f}")
    print(f"   R²   : {lstm_r2:.4f}")
    print(f"   Lookback window : {LOOKBACK} days")
else:
    # Fallback: MLP
    lstm_model = MLPClassifier if not TF_AVAILABLE else None
    mlp = MLPClassifier if not TF_AVAILABLE else None
    from sklearn.neural_network import MLPRegressor
    mlp = MLPRegressor(hidden_layer_sizes=(128, 64, 32), activation='relu',
                       max_iter=500, random_state=42, early_stopping=True)
    mlp.fit(X_train, y_train)
    mlp_pred = mlp.predict(X_test)
    lstm_pred   = mlp_pred
    lstm_actual = y_test
    lstm_rmse = np.sqrt(mean_squared_error(y_test, mlp_pred))
    lstm_mae  = mean_absolute_error(y_test, mlp_pred)
    lstm_r2   = r2_score(y_test, mlp_pred)
    print(f"\n📊 MLP (LSTM substitute) Results:")
    print(f"   RMSE : ${lstm_rmse:.2f}")
    print(f"   MAE  : ${lstm_mae:.2f}")
    print(f"   R²   : {lstm_r2:.4f}")

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  FINAL MODEL COMPARISON
# ══════════════════════════════════════════════════════════════════

all_models = {
    'Linear Regression': {'rmse': lr_rmse,   'mae': lr_mae,   'r2': lr_r2,   'pred': lr_pred,   'color': '#00d4ff'},
    'KNN':               {'rmse': knn_rmse,  'mae': knn_mae,  'r2': knn_r2,  'pred': knn_pred,  'color': '#a855f7'},
    'SVR':               {'rmse': svr_rmse,  'mae': svr_mae,  'r2': svr_r2,  'pred': svr_pred,  'color': '#f97316'},
    'Random Forest':     {'rmse': rf_rmse,   'mae': rf_mae,   'r2': rf_r2,   'pred': rf_pred,   'color': '#1D9E75'},
    'XGBoost':           {'rmse': xgb_rmse,  'mae': xgb_mae,  'r2': xgb_r2,  'pred': xgb_pred,  'color': '#f59e0b'},
    'LSTM / MLP':        {'rmse': lstm_rmse, 'mae': lstm_mae, 'r2': lstm_r2, 'pred': None,       'color': '#ec4899'},
}

names  = list(all_models.keys())
rmses  = [all_models[m]['rmse'] for m in names]
maes   = [all_models[m]['mae']  for m in names]
r2s    = [all_models[m]['r2']   for m in names]
colors = [all_models[m]['color'] for m in names]

fig = plt.figure(figsize=(20, 14))
fig.suptitle(f'📊 Final Model Comparison — {TICKER} Stock Price Prediction',
             fontsize=16, fontweight='bold', color='white', y=1.01)
gs  = gridspec.GridSpec(3, 2, figure=fig, hspace=0.5, wspace=0.3)

# 1. All predictions overlaid
ax1 = fig.add_subplot(gs[0, :])
ax1.plot(dates_test, y_test, color='white', linewidth=2, label='Actual Price', zorder=10)
for name in ['Linear Regression', 'KNN', 'SVR', 'Random Forest', 'XGBoost']:
    ax1.plot(dates_test, all_models[name]['pred'],
             color=all_models[name]['color'], linewidth=1.2,
             label=name, alpha=0.75)
ax1.set_title('All Models vs Actual Price', fontweight='bold', fontsize=13)
ax1.set_ylabel('Price (USD)')
ax1.legend(fontsize=9, ncol=3)
ax1.grid(True)

# 2. RMSE comparison
ax2 = fig.add_subplot(gs[1, 0])
bars = ax2.bar(names, rmses, color=colors, edgecolor='none', width=0.6)
best_rmse_idx = np.argmin(rmses)
bars[best_rmse_idx].set_edgecolor('#ffd700')
bars[best_rmse_idx].set_linewidth(3)
ax2.set_title('RMSE — Lower is Better\n(⭐ = Best model)', fontweight='bold')
ax2.set_ylabel('RMSE ($)')
ax2.tick_params(axis='x', rotation=25)
for bar, v in zip(bars, rmses):
    ax2.text(bar.get_x()+bar.get_width()/2, v+0.2, f'${v:.2f}',
             ha='center', fontsize=8, fontweight='bold')
ax2.grid(True, axis='y')

# 3. R² comparison
ax3 = fig.add_subplot(gs[1, 1])
bars3 = ax3.bar(names, r2s, color=colors, edgecolor='none', width=0.6)
best_r2_idx = np.argmax(r2s)
bars3[best_r2_idx].set_edgecolor('#ffd700')
bars3[best_r2_idx].set_linewidth(3)
ax3.set_title('R² Score — Higher is Better\n(1.0 = Perfect prediction)', fontweight='bold')
ax3.set_ylabel('R² Score')
ax3.set_ylim(min(r2s) - 0.05, 1.05)
ax3.tick_params(axis='x', rotation=25)
for bar, v in zip(bars3, r2s):
    ax3.text(bar.get_x()+bar.get_width()/2, v+0.002, f'{v:.4f}',
             ha='center', fontsize=8, fontweight='bold')
ax3.axhline(1.0, color='#ffd700', linestyle='--', alpha=0.5, linewidth=1)
ax3.grid(True, axis='y')

# 4. Summary table
ax4 = fig.add_subplot(gs[2, :])
ax4.axis('off')
table_data = [['Model', 'RMSE ($)', 'MAE ($)', 'R² Score', 'How it works (1 line)']]
descriptions = {
    'Linear Regression': 'Fits a weighted straight line through features',
    'KNN':               'Averages K most similar historical days',
    'SVR':               'Fits an ε-tube using RBF kernel in high dimensions',
    'Random Forest':     'Averages 200 random decision trees (Bagging)',
    'XGBoost':           'Sequential trees each correcting previous errors (Boosting)',
    'LSTM / MLP':        'Recurrent NN with memory gates for sequential patterns',
}
best_model = names[np.argmin(rmses)]
for name in names:
    m = all_models[name]
    marker = ' ⭐' if name == best_model else ''
    table_data.append([f'{name}{marker}', f'${m["rmse"]:.2f}', f'${m["mae"]:.2f}',
                       f'{m["r2"]:.4f}', descriptions[name]])
table = ax4.table(cellText=table_data[1:], colLabels=table_data[0],
                  loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1.1, 2.0)
# Header styling
for j in range(5):
    table[0, j].set_facecolor('#1e293b')
    table[0, j].set_text_props(color='white', fontweight='bold')
# Row styling
for i, name in enumerate(names):
    row_color = '#0f2027' if name == best_model else '#0f0f1a'
    for j in range(5):
        table[i+1, j].set_facecolor(row_color)
        table[i+1, j].set_text_props(color='#ccc')
ax4.set_title('Model Comparison Summary', fontweight='bold', pad=20)

plt.tight_layout()
plt.show()

print(f"\n{'='*55}")
print(f"  🏆 BEST MODEL: {best_model}")
print(f"     RMSE  : ${min(rmses):.2f}")
print(f"     R²    : {r2s[np.argmin(rmses)]:.4f}")
print(f"{'='*55}")